# Recurrent Neural Network を使った時系列予測

Dow Jones の株価指数について、RNN による時系列予測を行う。データセットは、https://github.com/mwaskom/seaborn-data にある `dowjones` を用いる。

In [ ]:
import seaborn as sns
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

In [ ]:
df = sns.load_dataset("dowjones")
df = df.set_index('Date')

data = df['Price'].values.astype(float)

data_mean = data.mean()
data_std = data.std()
normalized_data = (data - data_mean) / data_std

train_ratio = 0.6
val_ratio = 0.2

train_size = int(len(normalized_data) * train_ratio)
val_size = int(len(normalized_data) * val_ratio)
test_size = len(normalized_data) - train_size - val_size

train_data = normalized_data[:train_size]
val_data = normalized_data[train_size:train_size + val_size]
test_data = normalized_data[train_size + val_size:]

print(f"Total data: {len(normalized_data)}")
print(f"Train data: {len(train_data)}")
print(f"Validation data: {len(val_data)}")
print(f"Test data: {len(test_data)}")


def create_sequences(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 10
X_train, y_train = create_sequences(train_data, SEQ_LENGTH)
X_val, y_val = create_sequences(val_data, SEQ_LENGTH)
X_test, y_test = create_sequences(test_data, SEQ_LENGTH)

X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(-1)
X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(-1)
X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(-1)

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

train_dataset = TimeSeriesDataset(X_train_t, y_train_t)
val_dataset = TimeSeriesDataset(X_val_t, y_val_t)
test_dataset = TimeSeriesDataset(X_test_t, y_test_t)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)



class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, hn = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])
        return out

INPUT_SIZE = 1
HIDDEN_SIZE = 32
NUM_LAYERS = 1
OUTPUT_SIZE = 1

model = SimpleRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE)


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

NUM_EPOCHS = 10
train_losses = []
val_losses = []

print("Training Start")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for X_batch_val, y_batch_val in val_loader:
            outputs_val = model(X_batch_val)
            val_loss = criterion(outputs_val, y_batch_val)
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  Train Loss: {avg_train_loss:.4f},  Val Loss: {avg_val_loss:.4f}")

print("Training Complete")


def evaluate_model(model, loader):
    model.eval()
    predictions = []
    actuals = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            output = model(X_batch)
            predictions.extend(output.squeeze().tolist())
            actuals.extend(y_batch.squeeze().tolist())

    predictions_denorm = (np.array(predictions) * data_std) + data_mean
    actuals_denorm = (np.array(actuals) * data_std) + data_mean

    predictions_t = torch.tensor(predictions_denorm, dtype=torch.float32)
    actuals_t = torch.tensor(actuals_denorm, dtype=torch.float32)

    criterion = nn.MSELoss()
    test_loss = criterion(predictions_t, actuals_t).item()

    return test_loss, predictions_denorm, actuals_denorm


test_loss, test_predictions, test_actuals = evaluate_model(model, test_loader)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(test_actuals, label='Actual Price')
plt.plot(test_predictions, label='Predicted Price')
plt.xlabel('Time Step')
plt.ylabel('Dow Jones Price')
plt.title('RNN Prediction vs Actual (Test Set)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# ハイパーパラメータを変更して再学習
HIDDEN_SIZE = 64  # 隠れ層のサイズを増やす
NUM_LAYERS = 2    # RNN層を2層に
BATCH_SIZE = 32   # バッチサイズを増やす
NUM_EPOCHS = 20   # エポック数を増やす
LEARNING_RATE = 0.005  # 学習率を調整

# データローダーの再作成
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# モデルの再定義
model = SimpleRNN(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
val_losses = []

print("Training Start (Hyperparameter Tuning)")
for epoch in range(NUM_EPOCHS):
    model.train()
    total_train_loss = 0

    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for X_batch_val, y_batch_val in val_loader:
            outputs_val = model(X_batch_val)
            val_loss = criterion(outputs_val, y_batch_val)
            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]  Train Loss: {avg_train_loss:.4f},  Val Loss: {avg_val_loss:.4f}")

print("Training Complete (Hyperparameter Tuning)")

test_loss, test_predictions, test_actuals = evaluate_model(model, test_loader)
print(f"Test Loss after tuning: {test_loss:.4f}")